# Step 7 — enrich the catalog stops

Takes the qualified stops (step 5 ∪ step 6) and attaches everything the seed
catalog's seven columns cannot carry but the product needs:

- **`name_latin` / `name_ascii`** — the attributes the original ONTD export
  script (`getStations.py`) produced, with the same precedence: ONTD's curated
  Latin/ASCII columns where the stop came through ONTD; otherwise OSM's
  `name:sr-Latn` (RS/BA), `name:en`, `int_name`, then `transliterate` as the
  fallback for Cyrillic/Greek, `unidecode` for the ASCII fold.
- **`country_code`** plus the country's name in every member-organisation
  language, from pycountry's ISO 3166 translation catalogs — offline and
  reproducible.
- **`city`** — the municipality the stop belongs to (*Berlin Gesundbrunnen →
  Berlin*), resolved geographically against OSM `place=city|town` nodes, plus
  that city's name in every member language from the place node's own
  `name:<lang>` tags. This is what lets the frontend find München Hbf when an
  Italian user searches *Monaco*: exonyms are curated in OSM, so no machine
  translation is involved anywhere.

**Inputs:** `data/step5_JoinedNTStops.csv`, `data/step6_manual_additions.csv`,
`data/bahnhoefe_stops_sorted.csv` (ONTD), `data/step2_output_eu_stations.osm.pbf`
(full tags per station object), and `data/step7_place_nodes.csv` (fetched once
below via Overpass, then cached).

**Output:** `data/step7_enriched_stops.csv`, one row per catalog stop —
consumed by `step10_export_seed_stops.py` for the enrichment sidecar.

Station names themselves are deliberately **not** translated: stations rarely
have real exonyms, and machine-translating proper nouns is how a previous
attempt turned *Reading* and *Most* into common nouns. Search works through
the multilingual **city** and **country** columns beside the original,
Latin and ASCII station names.


## Imports, languages, paths

In [1]:
import csv
import gettext
import json
import math
import time
import urllib.request

import osmium
import pycountry
import regex
import transliterate
from unidecode import unidecode

from data_sources import DATA_DIR, ensure_local, local_input

# Member-organisation languages. Extending the catalog to another language is
# one entry here — every downstream column set is derived from it.
LANGS = ["en", "de", "fr", "nl", "it", "es", "pl"]

PBF_PATH = ensure_local("step2_output_eu_stations.osm.pbf")
PLACES_PATH = DATA_DIR / "step7_place_nodes.csv"
OUTPUT_PATH = DATA_DIR / "step7_enriched_stops.csv"

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

## Candidate stops

Union of the two qualification layers, step 5 first so its ONTD-backed row
wins where a stop qualifies both ways — the same precedence step 10 applies.


In [2]:
candidates = {}
with open(
    local_input("step5_JoinedNTStops.csv", "step5_JoinNTStopsWithOSM.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        candidates[row["osm_stop_id"]] = {
            "stop_id": row["osm_stop_id"],
            "name": row["osm_stop_name"] or row["ontd_name"],
            "country": row["ontd_country"].strip().upper(),
            "lat": float(row["osm_lat"]),
            "lon": float(row["osm_lon"]),
            "ontd_id": row["ontd_id"],
        }
with open(
    local_input("step6_manual_additions.csv", "step6_manual_additions.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        candidates.setdefault(
            row["stop_id"],
            {
                "stop_id": row["stop_id"],
                "name": row["stop_name"],
                "country": row["country"].strip().upper(),
                "lat": float(row["stop_lat"]),
                "lon": float(row["stop_lon"]),
                "ontd_id": "",
            },
        )
print(f"{len(candidates)} catalog stops to enrich")

1053 catalog stops to enrich


## OSM tags per stop

One osmium pass over the station extract, keeping only the tags the
enrichment uses. The extract carries **all** tags per object, so this replaces
any per-stop Overpass name lookup.


In [3]:
KEEP_TAGS = {"name", "int_name", "name:sr-Latn", "uic_ref"} | {
    f"name:{lang}" for lang in LANGS
}
_want = {
    prefix: {int(k[5:]) for k in candidates if k.startswith(f"osm:{prefix}")}
    for prefix in "nwr"
}
tags_of: dict[str, dict] = {}


class _StationTagHandler(osmium.SimpleHandler):
    def _grab(self, prefix, osm_id, tags):
        if osm_id in _want[prefix]:
            tags_of[f"osm:{prefix}{osm_id}"] = {
                t.k: t.v for t in tags if t.k in KEEP_TAGS
            }

    def node(self, o):
        self._grab("n", o.id, o.tags)

    def way(self, o):
        self._grab("w", o.id, o.tags)

    def relation(self, o):
        self._grab("r", o.id, o.tags)


_StationTagHandler().apply_file(str(PBF_PATH))
missing_pbf = sorted(set(candidates) - set(tags_of))
print(f"tags for {len(tags_of)} stops; {len(missing_pbf)} not in the extract")
if missing_pbf:
    # An id can legitimately be missing after an extract refresh deleted the
    # object; it still gets names via the fallback chain, so report, don't stop.
    for stop_id in missing_pbf[:10]:
        print(f"  {stop_id}  {candidates[stop_id]['name']}")

tags for 1053 stops; 0 not in the extract


## Latin and ASCII names

Precedence per `getStations.py`, the script that produced the ONTD export:
ONTD's curated columns win where the stop is ONTD-backed; OSM-only stops get
the same derivation ONTD's stops originally got.


In [4]:
_CYRILLIC = regex.compile(r"\p{Cyrillic}")
_GREEK = regex.compile(r"\p{Greek}")
# transliterate language pack per country, for names no OSM tag latinises.
_TRANSLIT_LANG = {
    "RS": "sr",
    "BG": "bg",
    "MK": "mk",
    "UA": "uk",
    "MD": "ru",
    "BA": "sr",
}

ontd_names = {}
with open(
    ensure_local("bahnhoefe_stops_sorted.csv"), encoding="utf-8-sig", newline=""
) as fh:
    for row in csv.DictReader(fh):
        ontd_names[row["ID"]] = (row["Name (Lateinisch)"], row["Name (ASCII)"])


def latinize(name: str, country: str, tags: dict) -> str:
    # Already Latin script: the name IS its own Latin form. Without this,
    # name:en turns transliteration into translation ("Köln Messe/Deutz" ->
    # "Cologne Trade Fair/Deutz"), which is not this column's job.
    if not _CYRILLIC.search(name) and not _GREEK.search(name):
        return name
    if country in ("RS", "BA") and _CYRILLIC.search(name) and tags.get("name:sr-Latn"):
        latin = tags["name:sr-Latn"]
    else:
        latin = tags.get("name:en") or tags.get("int_name") or name
    if _CYRILLIC.search(latin) and country in _TRANSLIT_LANG:
        latin = transliterate.translit(latin, _TRANSLIT_LANG[country], reversed=True)
    if _GREEK.search(latin):
        latin = transliterate.translit(latin, "el", reversed=True)
    return latin


unresolved_script = []
for stop in candidates.values():
    tags = tags_of.get(stop["stop_id"], {})
    ontd = ontd_names.get(stop["ontd_id"], ("", ""))
    # ONTD's curated Latin column only has a job when the name needs a script
    # conversion; for Latin-script names it can carry a *translation*
    # ("Cologne Trade Fair/Deutz"), which is not what this column is for.
    needs_transliteration = bool(
        _CYRILLIC.search(stop["name"]) or _GREEK.search(stop["name"])
    )
    if needs_transliteration and ontd[0].strip():
        stop["name_latin"], stop["name_ascii"] = ontd
    else:
        stop["name_latin"] = latinize(stop["name"], stop["country"], tags)
        stop["name_ascii"] = unidecode(stop["name_latin"])
    stop["uic_ref"] = tags.get("uic_ref", "")
    if _CYRILLIC.search(stop["name_latin"]) or _GREEK.search(stop["name_latin"]):
        unresolved_script.append((stop["stop_id"], stop["name"], stop["country"]))

# Predict-before-run anchor: with the 2026-08 data every name latinises.
print(f"{len(unresolved_script)} names still non-Latin (expected 0)")
for item in unresolved_script[:10]:
    print("  ", item)

0 names still non-Latin (expected 0)


## Display name

The raw OSM/ONTD `name` is not always what a passenger should see. Two
patterns get normalised, everything else stays untouched:

- **Official bilingual names** — `Koper / Capodistria`, `Fribourg/Freiburg`,
  `Donostia / San Sebastián`, `Cottbus Hauptbahnhof / Chóśebuz głowne
  dwórnišćo`... The slash-separated second language is real signage, but as a
  display name it doubles the length and breaks search expectations; the first
  segment is the primary local form, and the other language belongs in the
  multilingual columns. Only `/` splits — `Latour-de-Carol - Enveitg` and
  `Calalzo - Pieve di Cadore - Cortina` are single official names for
  stations serving several places, so ` - ` never does.
- **Explicit overrides** — where a rule cannot decide. `Milano Porta
  Garibaldi (superficie)` is the correct *object* (night trains use the
  surface tracks; ONTD has no unqualified variant) but the qualifier is not
  passenger-facing. Overrides are validated against the candidate set so a
  stale id fails loudly.

Latin and ASCII names are recomputed from the display name where it changed,
so all three stay consistent.


In [5]:
# Display-name corrections, explicit per stop. Two groups, one mechanism:
# bilingual signage names (Koper / Capodistria) shortened to the primary
# form, and platform-group qualifiers ONTD carried into the name (Milano
# Porta Garibaldi (superficie)). No splitting heuristic: a slash or dash
# cannot tell "Fribourg/Freiburg" (two languages, one city) from
# "Siegburg/Bonn" or "Barcelona - Sants" (one official name) — that is
# language knowledge, so it stays a curated decision. The full original
# name remains in the step 5/6 outputs and in OSM; name_latin/name_ascii
# are re-derived from the displayed form so the three stay consistent.
DISPLAY_NAME_OVERRIDES = {
    # bilingual -> primary
    "osm:n1837570219": "Koper",  # / Capodistria
    "osm:n5526332068": "Mo i Rana",  # / Måefie (Southern Sami)
    "osm:n5526331885": "Bodø",  # - Bådåddjo (Lule Sami)
    # Fribourg/Freiburg (osm:n3081154375) and Donostia / San Sebastián
    # (osm:n6080884392) were dropped on 2026-08-28: both were step 5 stops
    # under the frozen schedule export, and no active night train in the ONTD
    # workbook calls at either. Re-add here, with the then-current OSM id, if
    # step 6 admits them as FUA additions.
    "osm:n2599505466": "Cottbus Hauptbahnhof",  # / Chóśebuz głowne dwórnišćo
    "osm:n4189000814": "Flensburg",  # / Flensborg
    "osm:n4586092220": "Oviedo",  # / Uviéu
    "osm:n11757382798": "Pamplona",  # / Iruña
    "osm:n17401552": "Bruxelles-Midi",  # - Brussel-Zuid
    "osm:n66180313": "Bruxelles-Nord",  # - Brussel-Noord
    "osm:n1455004982": "Bolzano",  # - Bozen
    # platform-group qualifier from ONTD
    "osm:n11635678177": "Milano Porta Garibaldi",  # (superficie)
}

# An id here can go stale two ways, and they need different fixes: the stop
# left the catalog, or step 5 re-matched it to a different OSM object for the
# same station (its matcher improves, and the object it picks moves with it).
# The second is far more common, so the failure names the likely replacement
# rather than only the missing id.
unknown_overrides = sorted(set(DISPLAY_NAME_OVERRIDES) - set(candidates))
if unknown_overrides:
    hints = []
    for stop_id in unknown_overrides:
        wanted = DISPLAY_NAME_OVERRIDES[stop_id].casefold()
        same_name = sorted(
            f"{other} ({stop['name']})"
            for other, stop in candidates.items()
            if wanted in stop["name"].casefold()
        )
        hints.append(
            f"    {stop_id} -> {DISPLAY_NAME_OVERRIDES[stop_id]!r}: "
            + (
                "now " + ", ".join(same_name[:3])
                if same_name
                else "no stop of that name in the catalog — it dropped out"
            )
        )
    raise KeyError(
        f"{len(unknown_overrides)} DISPLAY_NAME_OVERRIDES id(s) are not in the "
        "catalog. Repoint them if the station is still there under a new OSM "
        "object, remove them if it is gone:\n" + "\n".join(hints)
    )

renamed = []
for stop_id, shown in DISPLAY_NAME_OVERRIDES.items():
    stop = candidates[stop_id]
    renamed.append((stop["name"], shown))
    stop["name"] = shown
    stop["name_latin"] = latinize(shown, stop["country"], tags_of.get(stop_id, {}))
    stop["name_ascii"] = unidecode(stop["name_latin"])

print(f"{len(renamed)} display names overridden:")
for old, new in sorted(renamed):
    print(f"  {old[:52]:54} -> {new}")

# Review net for future entrants: separator-carrying names not covered above.
# Compound official names (Barcelona - Sants) are correct and stay; a NEW
# bilingual name should be added to the dict, so it is listed here once.
uncovered = sorted(
    stop["name"]
    for stop in candidates.values()
    if ("/" in stop["name"] or " - " in stop["name"])
    and stop["stop_id"] not in DISPLAY_NAME_OVERRIDES
)
print(
    f"\n{len(uncovered)} compound names left as-is (add to the dict only if bilingual):"
)
for name in uncovered:
    print(f"  {name}")

11 display names overridden:
  Bodø - Bådåddjo                                        -> Bodø
  Bolzano - Bozen                                        -> Bolzano
  Bruxelles-Midi - Brussel-Zuid                          -> Bruxelles-Midi
  Bruxelles-Nord - Brussel-Noord                         -> Bruxelles-Nord
  Cottbus Hauptbahnhof / Chóśebuz głowne dwórnišćo       -> Cottbus Hauptbahnhof
  Flensburg / Flensborg                                  -> Flensburg
  Koper / Capodistria                                    -> Koper
  Milano Porta Garibaldi (superficie)                    -> Milano Porta Garibaldi
  Mo i Rana / Måefie                                     -> Mo i Rana
  Oviedo / Uviéu                                         -> Oviedo
  Pamplona / Iruña                                       -> Pamplona

9 compound names left as-is (add to the dict only if bilingual):
  Bretenoux - Biars
  Chambéry - Challes-les-Eaux
  Latour-de-Carol - Enveitg
  Lisboa - Oriente
  Porto - Campanhã


## Country names per language

pycountry ships the ISO 3166 gettext catalogs — offline, reproducible, and
already a project dependency. `common_name` preferred where ISO's formal name
is not what anyone types (*Moldova, Republic of*). Kosovo is not in ISO 3166,
so `XK` is a manual row.


In [6]:
_translations = {
    lang: gettext.translation("iso3166-1", pycountry.LOCALES_DIR, languages=[lang])
    for lang in LANGS
    if lang != "en"
}

COUNTRY_NAME_OVERRIDES = {
    "XK": {  # not in ISO 3166
        "en": "Kosovo",
        "de": "Kosovo",
        "fr": "Kosovo",
        "nl": "Kosovo",
        "it": "Kosovo",
        "es": "Kosovo",
        "pl": "Kosowo",
    },
}


def country_names(alpha2: str) -> dict[str, str]:
    if alpha2 in COUNTRY_NAME_OVERRIDES:
        return COUNTRY_NAME_OVERRIDES[alpha2]
    country = pycountry.countries.get(alpha_2=alpha2)
    if country is None:
        return {lang: "" for lang in LANGS}
    base = getattr(country, "common_name", None) or country.name
    return {
        lang: base if lang == "en" else _translations[lang].gettext(base)
        for lang in LANGS
    }


_country_cache = {
    c: country_names(c) for c in {s["country"] for s in candidates.values()}
}
blank = sorted(c for c, names in _country_cache.items() if not names["en"])
if blank:
    raise ValueError(f"country code(s) with no name: {blank} — add an override above")
print({c: n["it"] for c, n in sorted(_country_cache.items())[:8]})

{'AL': 'Albania', 'AT': 'Austria', 'BA': 'Bosnia-Erzegovina', 'BE': 'Belgio', 'BG': 'Bulgaria', 'CH': 'Svizzera', 'CZ': 'Cechia', 'DE': 'Germania'}


## Place nodes — fetch once, then cached

`place=city|town` nodes per country from Overpass, with their `name:<lang>`
tags and `population`. The country/area table is the one `getStations.py`
used for the ONTD export, filtered to the countries the catalog actually
contains. One request per country, mirroring step 3a's Overpass conventions
(retry on transient errors, report failures, stay resumable). The result is
cached in `data/step7_place_nodes.csv`; delete the file to re-fetch.


In [7]:
# OSM area ids per country — from getStations.py, the ONTD export script.
OVERPASS_AREAS = {
    "BA": 3602528142,
    "HR": 3600214885,
    "ME": 3600053296,
    "RS": 3601741311,
    "SI": 3600218657,
    "MK": 3600053293,
    "AL": 3600053292,
    "XK": 3602088990,
    "BG": 3600186382,
    "GR": 3600192307,
    "TR": 3600174737,
    "RO": 3600090689,
    "MD": 3600058974,
    "UA": 3600060199,
    "HU": 3600021335,
    "SK": 3600014296,
    "CZ": 3600051684,
    "PL": 3600049715,
    "LT": 3600072596,
    "LV": 3600072594,
    "EE": 3600079510,
    "FI": 3600054224,
    "NO": 3602978650,
    "SE": 3600052822,
    "DK": 3600050046,
    "IE": 3600062273,
    "GB": 3600062149,
    "FR": 3602202162,
    "ES": 3601311341,
    "PT": 3600295480,
    "IT": 3600365331,
    "CH": 3600051701,
    "AT": 3600016239,
    "BE": 3600052411,
    "NL": 3602323309,
    "LU": 3602171347,
    "MC": 3601124039,
    "LI": 3601155955,
    "DE": 3600051477,
}

_PLACE_FIELDS = ["osm_id", "country", "place", "name", "population", "lat", "lon"] + [
    f"name_{lang}" for lang in LANGS
]


def fetch_place_nodes(countries: list[str]) -> tuple[list[dict], list[str]]:
    """Fetch city/town place nodes for the given countries. Returns
    (rows, failed_countries). Two public Overpass instances are tried in
    rotation — rate limits and timeouts on one are routine."""
    # Three public instances, rotated. 504 (overpass-api.de) and 500 (kumi)
    # are their overloaded responses, they cluster in time, and they are not
    # about the query — so more instances and longer waits beat fewer, faster
    # retries.
    endpoints = [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://overpass.openstreetmap.fr/api/interpreter",
    ]
    rows, failed = [], []
    for country in countries:
        query = (
            # 300 not 600: some instances cap the server-side timeout and a
            # smaller ask schedules sooner on a busy box. City/town nodes for
            # one country complete in seconds when the query runs at all.
            f"[out:json][timeout:300];area({OVERPASS_AREAS[country]})->.a;"
            f'node["place"~"^(city|town)$"](area.a);out body;'
        )
        for attempt in (1, 2, 3, 4, 5, 6):
            url = endpoints[(attempt - 1) % len(endpoints)]
            try:
                req = urllib.request.Request(
                    url,
                    data=query.encode(),
                    headers={"User-Agent": "night-train-target-network stop pipeline"},
                )
                with urllib.request.urlopen(req, timeout=600) as resp:
                    elements = json.load(resp).get("elements", [])
                break
            except Exception as exc:  # transient Overpass errors are normal
                print(
                    f"  {country}: attempt {attempt} via {url.split('/')[2]} failed ({exc})"
                )
                time.sleep(30 * attempt)
        else:
            failed.append(country)
            continue
        count = 0
        for el in elements:
            tags = el.get("tags", {})
            if not tags.get("name"):
                continue
            row = {
                "osm_id": el["id"],
                "country": country,
                "place": tags.get("place", ""),
                "name": tags["name"],
                "population": tags.get("population", ""),
                "lat": el["lat"],
                "lon": el["lon"],
            }
            for lang in LANGS:
                row[f"name_{lang}"] = tags.get(f"name:{lang}", "")
            rows.append(row)
            count += 1
        print(f"  {country}: {count} places")
        time.sleep(5)  # be polite to the public instances
    return rows, failed


# The cache is per COUNTRY, not per file. A partial fetch is routine (Overpass
# rate-limits and times out), so completeness is judged by which countries the
# file covers: a re-run fetches exactly the gaps and appends. This replaced an
# existence check that made the fetch's own "re-run this cell to fill them in"
# advice a no-op — the file existed, so the re-run silently skipped, and NL sat
# empty while every Dutch stop "resolved no city". Delete the file to force a
# full re-fetch.
needed = sorted({s["country"] for s in candidates.values()} & set(OVERPASS_AREAS))
outside = sorted({s["country"] for s in candidates.values()} - set(OVERPASS_AREAS))
if outside:
    print(f"  no Overpass area for {outside} — their stops resolve no city")

places = []
if PLACES_PATH.is_file():
    with open(PLACES_PATH, encoding="utf-8-sig", newline="") as fh:
        places = list(csv.DictReader(fh))

covered = {row["country"] for row in places}
missing = [c for c in needed if c not in covered]
if missing:
    print(f"fetching {len(missing)} countries not in the cache: {missing}")
    fetched, failed = fetch_place_nodes(missing)
    places.extend(fetched)
    with open(PLACES_PATH, "w", encoding="utf-8-sig", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=_PLACE_FIELDS)
        writer.writeheader()
        writer.writerows(places)
    if failed:
        raise RuntimeError(
            f"place fetch failed for {failed} after 6 attempts each — their "
            "stops would silently resolve no city. Re-run this cell; only the "
            "failed countries are re-fetched."
        )

empty = [c for c in needed if not any(r["country"] == c for r in places)]
if empty:
    raise RuntimeError(
        f"cache covers {sorted(covered)} but holds ZERO places for {empty} — "
        "a failed fetch was cached. Delete data/step7_place_nodes.csv rows for "
        "those countries (or the whole file) and re-run."
    )

for row in places:
    row["lat"], row["lon"] = float(row["lat"]), float(row["lon"])
    try:
        row["population"] = int(row["population"])
    except ValueError:
        row["population"] = 0
print(f"{len(places)} place nodes loaded, {len(covered | set(missing))} countries")

17355 place nodes loaded, 36 countries


## City per stop

Nearest-place resolution with a hierarchy: a `city` within 20 km beats a
`town` within 10 km, because suburb stations sit closer to their own suburb's
node than to the city they belong to — *Berlin-Spandau is Berlin, not
Falkensee*. Within a tier, larger population wins ties on distance-adjusted
score. Everything is written for review; unresolved stops are listed, and
misresolutions get one line in `CITY_OVERRIDES` (checked against the place
list, so a typo fails loudly).


In [8]:
CITY_RADIUS_KM = 20.0
TOWN_RADIUS_KM = 10.0

# stop_id -> place node osm_id, for the cases geography gets wrong.
CITY_OVERRIDES: dict[str, int] = {}


def distance_km(lat1, lon1, lat2, lon2):
    # Equirectangular approximation — fine at city scale.
    mean_lat = math.radians((lat1 + lat2) / 2)
    dx = math.radians(lon2 - lon1) * math.cos(mean_lat)
    dy = math.radians(lat2 - lat1)
    return math.hypot(dx, dy) * 6371.0


places_by_id = {p["osm_id"]: p for p in places}
unknown_overrides = {
    stop_id: place_id
    for stop_id, place_id in CITY_OVERRIDES.items()
    if str(place_id) not in places_by_id and place_id not in places_by_id
}
if unknown_overrides:
    raise KeyError(f"CITY_OVERRIDES point at unknown place nodes: {unknown_overrides}")


def resolve_city(stop) -> dict | None:
    override = CITY_OVERRIDES.get(stop["stop_id"])
    if override is not None:
        return places_by_id.get(override) or places_by_id[str(override)]
    best = None
    for tier, radius in (("city", CITY_RADIUS_KM), ("town", TOWN_RADIUS_KM)):
        candidates_in_tier = []
        for p in places:
            if p["place"] != tier:
                continue
            d = distance_km(stop["lat"], stop["lon"], p["lat"], p["lon"])
            if d <= radius:
                # Population dominates at equal distance; distance dominates
                # between a near small town and a far one.
                candidates_in_tier.append((d - math.log10(p["population"] + 10), d, p))
        if candidates_in_tier:
            best = min(candidates_in_tier)[2]
            break
    return best


unresolved = []
for stop in candidates.values():
    place = resolve_city(stop)
    stop["city"] = place["name"] if place else ""
    stop["city_osm_id"] = place["osm_id"] if place else ""
    for lang in LANGS:
        stop[f"city_{lang}"] = (
            (place.get(f"name_{lang}") or place["name"]) if place else ""
        )
    if place is None:
        unresolved.append((stop["stop_id"], stop["name"], stop["country"]))

print(f"{len(unresolved)} stops resolve no city — rural halts are expected here")
for item in unresolved[:20]:
    print("  ", item)

30 stops resolve no city — rural halts are expected here
   ('osm:n2322386861', 'Синдел Разпределителна', 'BG')
   ('osm:n1157648088', 'Поповица', 'BG')
   ('osm:n13042236249', 'Вакарел', 'BG')
   ('osm:n1489590945', 'Безименна', 'BG')
   ('osm:n2319266710', 'Богданци', 'BG')
   ('osm:n700937220', 'Телиш', 'BG')
   ('osm:n1829927002', 'Bánovce nad Ondavou', 'SK')
   ('osm:n9040800059', 'Štrba', 'SK')
   ('osm:n9993048555', 'Lőkösháza', 'HU')
   ('osm:n2155870540', 'Deda', 'RO')
   ('osm:n6749038713', 'Crianlarich', 'GB')
   ('osm:n3074774942', 'Göschenen', 'CH')
   ('osm:n327287949', 'St. Anton am Arlberg', 'AT')
   ('osm:n463092859', 'Langen am Arlberg', 'AT')
   ('osm:n1333665650', 'Sargans', 'CH')
   ('osm:n4629874447', 'Baqël', 'AL')
   ('osm:n996203371', 'Fårevejle', 'SE')
   ('osm:n13715292382', 'Duved', 'SE')
   ('osm:n257391370', 'Saint-Denis-près-Martel', 'FR')
   ('osm:n7389146997', 'Bretenoux - Biars', 'FR')


## Write

One row per catalog stop. Country names attach here so the file is complete
per stop and step 10 can pass it through without joins.


In [9]:
fieldnames = (
    ["stop_id", "stop_name", "name_latin", "name_ascii", "uic_ref", "country_code"]
    + [f"country_{lang}" for lang in LANGS]
    + ["city", "city_osm_id"]
    + [f"city_{lang}" for lang in LANGS]
)

with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    for stop in sorted(candidates.values(), key=lambda s: (s["country"], s["name"])):
        row = {
            "stop_id": stop["stop_id"],
            "stop_name": stop["name"],
            "name_latin": stop["name_latin"],
            "name_ascii": stop["name_ascii"],
            "uic_ref": stop["uic_ref"],
            "country_code": stop["country"],
            "city": stop["city"],
            "city_osm_id": stop["city_osm_id"],
        }
        row.update(
            {f"country_{lang}": _country_cache[stop["country"]][lang] for lang in LANGS}
        )
        row.update({f"city_{lang}": stop[f"city_{lang}"] for lang in LANGS})
        writer.writerow(row)

print(f"{len(candidates)} enriched stops -> {OUTPUT_PATH.name}")

# The point of the exercise, spot-checked: an Italian search for "Monaco"
# has to reach München's stops through city_it.
munich = [s for s in candidates.values() if s.get("city") == "München"]
if munich:
    print("München stops city_it:", {s["name"]: s["city_it"] for s in munich})

1053 enriched stops -> step7_enriched_stops.csv
München stops city_it: {'München Ost': 'Monaco di Baviera', 'München Hauptbahnhof': 'Monaco di Baviera', 'München Süd': 'Monaco di Baviera', 'München-Pasing': 'Monaco di Baviera'}
